# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabeeljohn2/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import os, subprocess, json
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("/content/flyrank-ml-internship"):
        subprocess.run(["git","clone","https://github.com/tabeeljohn2/flyrank-ml-internship.git","/content/flyrank-ml-internship"])
    os.chdir("/content/flyrank-ml-internship")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

d = df[df["impressions_prev_30d"] >= 50].copy()
d["ratio"] = d["impressions_last_30d"] / d["impressions_prev_30d"]
d["declined"] = (d["ratio"] < 0.7).astype(int)
NUM = ["days_since_last_update","content_age_days","word_count","search_volume","competition","cpc"]
CAT = ["content_type","main_intent"]
FEATURES = NUM + CAT
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
models = {
 "Logistic Regression": Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
 "Random Forest": Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50, random_state=42, n_jobs=-1))]),
}
print(d.shape, "base rate", round(d["declined"].mean(), 3))

In [ ]:
d["p_decline"] = cross_val_predict(models["Logistic Regression"], d[FEATURES], d["declined"],
                                   groups=d["client_id"], cv=GroupKFold(n_splits=5),
                                   method="predict_proba")[:,1]
oof_auc = roc_auc_score(d["declined"], d["p_decline"])
print("out-of-fold AUC (unseen clients):", round(oof_auc, 3), "| base rate:", round(d["declined"].mean(), 3))

In [ ]:
vis = d["impressions_90d"]
d["pos_bucket"] = pd.qcut(d["avg_position"], 5, duplicates="drop")
d["ctr_pct_in_pos"] = d.groupby("pos_bucket", observed=True)["ctr"].rank(pct=True)

low_vis   = vis < vis.quantile(0.25)
low_ctr   = (d["avg_position"] <= 10) & (d["ctr_pct_in_pos"] < 0.25)
high_risk = d["p_decline"] >= d["p_decline"].quantile(0.80)
striking  = (d["avg_position"] > 10) & (d["avg_position"] <= 20)

conds = [low_vis, low_ctr, high_risk, striking]
codes = ["LOW_VISIBILITY","LOW_CTR_FOR_POSITION","HIGH_DECLINE_RISK","STRIKING_DISTANCE"]
acts  = ["NO_ACTION","REVIEW_TITLE_SNIPPET","REVIEW_FOR_REFRESH","REVIEW_CONTENT_GAPS"]
d["reason_code"] = np.select(conds, codes, default="STABLE")
d["action"]      = np.select(conds, acts,  default="MONITOR")
d["actionable"]  = d["action"].str.startswith("REVIEW").astype(int)

queue = d.sort_values(["actionable","p_decline"], ascending=[False,False]).reset_index(drop=True)
queue["rank"] = queue.index + 1
print(queue["reason_code"].value_counts())
cols = ["rank","content_id","p_decline","reason_code","action","content_age_days","days_since_last_update","impressions_90d","avg_position","ctr"]
queue.head(10)[cols].round(3)

| Archetype | Reason code | Action (human reviews first) | Basis |
|---|---|---|---|
| Visible page, top-10 position, CTR in the bottom quarter for its position | LOW_CTR_FOR_POSITION | Review title and snippet | Position vs CTR verdict: CONFIRMED (Week 4) |
| Page in the top 20% of modeled decline risk | HIGH_DECLINE_RISK | Review for refresh | Grouped-validated model, directional only (Week 5/6) |
| Position 11 to 20 | STRIKING_DISTANCE | Review for content gaps | Rule of thumb, not validated here |
| Bottom quarter of impressions | LOW_VISIBILITY | No action | Cost: editor time is better spent elsewhere |
| Everything else | STABLE | Monitor | None |

In [ ]:
age = d.assign(b=pd.qcut(d["content_age_days"], 5, duplicates="drop")).groupby("b", observed=True)["declined"].agg(n="size", decline_rate="mean").round(3)
stale = d.assign(b=pd.qcut(d["days_since_last_update"], 5, duplicates="drop")).groupby("b", observed=True)["declined"].agg(n="size", decline_rate="mean").round(3)
print(age); print(stale)

fig, ax = plt.subplots(figsize=(6,3.5))
ax.bar(range(len(age)), age["decline_rate"])
ax.set_xticks(range(len(age))); ax.set_xticklabels([str(i) for i in age.index], rotation=30, ha="right")
ax.axhline(d["declined"].mean(), ls="--"); ax.set_ylabel("share declined (observed)")
ax.set_title("Decline rate by content age (dashed = overall)")
plt.tight_layout(); os.makedirs("work/figures", exist_ok=True)
plt.savefig("work/figures/decline_by_age.png", dpi=150); plt.show()

Decay / refresh insight (observed, directional). The decline rate by content-age bucket was 0.594, 0.691, 0.562, 0.446 and 0.308 (youngest to oldest), so older pages declined less often than younger ones in this sample, though the youngest bucket sits below the second. Decline rate by days since last update showed no steady pattern (0.597, 0.287, 0.530, 0.791; the last bucket has only 115 pages), and Week 4's Signal 1 was MIXED. Observed: decline in this data tracks page age more than time since last update. This data does not measure whether a refresh changes the outcome, so I make no claim that refreshing reduces decline.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
elig = queue[queue["actionable"]==1]
rows = []
for N in [50, 100, 200, 500]:
    t = elig.head(N)
    lost = (t["impressions_prev_30d"] - t["impressions_last_30d"]).clip(lower=0).sum()
    rows.append({"weekly_capacity_N": N,
                 "decline_rate_top_N": round(t["declined"].mean(), 3),
                 "eligible_base_rate": round(elig["declined"].mean(), 3),
                 "editor_hours_ASSUMED_2h_per_page": 2*N,
                 "impressions_lost_last30d_in_picks": int(lost)})
cost_table = pd.DataFrame(rows); print(cost_table)
print(elig.head(200)["client_id"].value_counts(normalize=True).head(5).round(2))


Intended use: decision-support for a human editor who triages which pages to look at first each week. It is a research prototype and is not a production system.

Limits:
- Signal strength: in grouped validation (clients held out), Logistic Regression had a mean AUC of 0.583 across 10 splits, ranging from 0.455 to 0.876. The 5-fold out-of-fold AUC in this notebook is 0.611, and the difference comes from the different splits. Both are directionally above the Week-4 rule (0.420), but weak and unstable.
- The model's signal appears to rest on one feature (AUC 0.475 without content_age_days).
- The label is an observed drop in impressions (last 30 days below 70% of the previous 30 days), measured over one window and only for pages with at least 50 impressions in the previous 30 days. It is not the effect of any action. Nothing here shows that a refresh would help.
- Only 30 clients, and the top five hold about 79% of the top-200 picks (0.30, 0.18, 0.17, 0.08, 0.06). Results may not transfer to a new client.
- The 0.7 threshold, the archetype cut-offs (including the top 20% risk cut-off) and the 2 editor-hours per page are my choices or assumptions.
- No seasonality, no time-based validation, and no future-window check.
- Lift over the base rate is concentrated in the top 100 to 200 picks (0.660 and 0.665 vs a 0.538 base). It is absent at the top 50 (0.540) and fades at 500 (0.574). These samples are small.
- word_count is missing for 29.5% of pages and is median-imputed, so any effect of that feature is weakly measured.
- "Impressions lost in picks" is the observed drop in the last 30 days for the selected pages. It is not recoverable upside.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Human-review rules:
1. An editor reads every REVIEW_* page before any change. The queue is a reading list, not an instruction.
2. The editor checks the page's recent history for a seasonal pattern, a tracking gap, or a recent update that the data does not show.
3. For LOW_CTR_FOR_POSITION, the editor also checks the search results page for features that explain a low CTR.
4. The editor records the decision (act, skip, or unsure) and a reason, so the next review can learn from it.
5. Pages with small impression bases get extra caution, because ratios there are noisy.

Do NOT automate:
- Publishing, rewriting or deleting pages, or changing titles and redirects, from a score.
- Promising a client that a refresh will recover traffic.
- Treating the score as a cause ("this page declined because it is old").
- Applying the queue to a new client without re-validating on held-out clients.
- Ranking or comparing editors or clients with these scores.
- Using the Week-4 rule, which scored below chance.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Intended use: decision-support for a human editor who triages which pages to look at first each week. It is a research prototype and is not a production system.

Limits:
- Signal strength: in grouped validation (clients held out), Logistic Regression had a mean AUC of 0.583 across 10 splits, ranging from 0.455 to 0.876. Directionally above the Week-4 rule (0.420), but weak and unstable.
- The model's signal appears to rest on one feature (AUC 0.475 without content_age_days).
- The label is an observed drop in impressions (last 30 days below 70% of the previous 30 days), measured over one window. It is not the effect of any action. Nothing here shows that a refresh would help.
- Only 30 clients, and the top five hold about 79% of the top-200 picks (0.30, 0.18, 0.17, 0.08, 0.06). Results may not transfer to a new client.
- The 0.7 threshold, the archetype cut-offs and the 2 editor-hours per page are my choices or assumptions.
- No seasonality, no time-based validation, and no future-window check.
- Lift over the base rate is concentrated in the top 100 to 200 picks (0.660 and 0.665 vs a 0.538 base). It is absent at the top 50 (0.540) and fades at 500 (0.574). These samples are small.
- word_count is missing for 29.5% of pages and is median-imputed, so any effect of that feature is weakly measured.
- "Impressions lost in picks" is the observed drop in the last 30 days for the selected pages. It is not recoverable upside.

In [ ]:
missing = d[FEATURES].isna().mean().round(3).to_dict()
print(missing)

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
os.makedirs("work/outputs", exist_ok=True)
export_cols = ["rank","client_id","content_id","p_decline","reason_code","action",
               "content_type","content_age_days","days_since_last_update","impressions_90d","avg_position","ctr"]
queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
queue[export_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)  # name from the original brief

metrics = {"rows": int(len(d)), "clients": int(d["client_id"].nunique()),
           "decline_base_rate": round(float(d["declined"].mean()), 3),
           "oof_auc_logistic_grouped_5fold": round(float(oof_auc), 3),
           # hard-coded from the w06 grouped validation (10 splits), not computed in this notebook
           "week6_grouped_auc_mean_10_splits": {"rule": 0.420, "logistic": 0.583, "random_forest": 0.585},
           "action_counts": queue["action"].value_counts().to_dict(),
           "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
           "feature_missing_rates": missing,
           "cost_table": cost_table.to_dict(orient="records"),
           "assumptions": {"editor_hours_per_page": 2, "decline_threshold": 0.7}}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(os.listdir("work/outputs"), os.listdir("work/figures"))

## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅ ] No client names, URLs, or private queries anywhere
- [ ✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.